In [5]:
import math
import numpy as np


# ============================================================
# EXERCÍCIO 3 - BALANCEAMENTO DE CARGA EM SERVIDORES
# ============================================================

np.random.seed(2026)

TAREFAS = np.array([
    12, 35, 40, 8, 15,
    22, 19, 45, 60, 31,
    14, 28, 50, 18, 25,
    33, 42, 10, 5, 29
])

NUM_SERVIDORES = 4

TAM_POPULACAO = 120
NUM_GERACOES = 400
TAXA_MUTACAO = 0.08
TAMANHO_TORNEIO = 3


def calcular_cargas(individuo):
    cargas = np.zeros(
        NUM_SERVIDORES,
        dtype=int
    )

    for indice_tarefa, servidor in enumerate(individuo):
        cargas[servidor] += TAREFAS[indice_tarefa]

    return cargas


def calcular_makespan(individuo):
    cargas = calcular_cargas(individuo)

    return int(
        np.max(cargas)
    )


def criar_individuo():
    return np.random.randint(
        0,
        NUM_SERVIDORES,
        size=len(TAREFAS)
    )


def selecao_torneio(populacao, fitness):
    participantes = np.random.choice(
        len(populacao),
        TAMANHO_TORNEIO,
        replace=False
    )

    melhor = min(
        participantes,
        key=lambda i: fitness[i]
    )

    return populacao[melhor]


def crossover(pai1, pai2):
    ponto = np.random.randint(
        1,
        len(TAREFAS)
    )

    return np.concatenate((
        pai1[:ponto],
        pai2[ponto:]
    ))


def mutacao(individuo):
    filho = individuo.copy()

    for i in range(len(filho)):
        if np.random.rand() < TAXA_MUTACAO:
            filho[i] = np.random.randint(
                0,
                NUM_SERVIDORES
            )

    return filho


populacao = [
    criar_individuo()
    for _ in range(TAM_POPULACAO)
]

melhor_individuo_global = None
melhor_makespan_global = math.inf


for _ in range(NUM_GERACOES):
    fitness = [
        calcular_makespan(individuo)
        for individuo in populacao
    ]

    melhor_idx = int(
        np.argmin(fitness)
    )

    if fitness[melhor_idx] < melhor_makespan_global:
        melhor_makespan_global = fitness[melhor_idx]

        melhor_individuo_global = (
            populacao[melhor_idx].copy()
        )

    # Elitismo
    nova_populacao = [
        melhor_individuo_global.copy()
    ]

    while len(nova_populacao) < TAM_POPULACAO:
        pai1 = selecao_torneio(
            populacao,
            fitness
        )

        pai2 = selecao_torneio(
            populacao,
            fitness
        )

        filho = crossover(
            pai1,
            pai2
        )

        filho = mutacao(
            filho
        )

        nova_populacao.append(
            filho
        )

    populacao = nova_populacao


cargas_finais = calcular_cargas(
    melhor_individuo_global
)

limite_inferior = math.ceil(
    int(np.sum(TAREFAS)) / NUM_SERVIDORES
)


print("=" * 65)
print("EXERCÍCIO 3 - BALANCEAMENTO DE CARGA")
print("=" * 65)

print(
    "Indivíduo encontrado:",
    melhor_individuo_global.tolist()
)

for servidor in range(NUM_SERVIDORES):
    indices = [
        i
        for i, servidor_alocado
        in enumerate(melhor_individuo_global)
        if servidor_alocado == servidor
    ]

    tempos = [
        int(TAREFAS[i])
        for i in indices
    ]

    print(
        f"\nServidor {servidor}:"
    )

    print(
        f"  Tarefas (índices): {indices}"
    )

    print(
        f"  Tempos: {tempos}"
    )

    print(
        f"  Carga total: {int(cargas_finais[servidor])} s"
    )

print(
    f"\nMakespan encontrado: "
    f"{melhor_makespan_global} s"
)

print(
    f"Limite inferior teórico: "
    f"{limite_inferior} s"
)

if melhor_makespan_global == limite_inferior:
    print(
        "O makespan encontrado atingiu o menor valor "
        "teoricamente possível."
    )


EXERCÍCIO 3 - BALANCEAMENTO DE CARGA
Indivíduo encontrado: [3, 2, 0, 2, 0, 2, 1, 3, 3, 2, 1, 1, 0, 3, 0, 1, 1, 2, 0, 2]

Servidor 0:
  Tarefas (índices): [2, 4, 12, 14, 18]
  Tempos: [40, 15, 50, 25, 5]
  Carga total: 135 s

Servidor 1:
  Tarefas (índices): [6, 10, 11, 15, 16]
  Tempos: [19, 14, 28, 33, 42]
  Carga total: 136 s

Servidor 2:
  Tarefas (índices): [1, 3, 5, 9, 17, 19]
  Tempos: [35, 8, 22, 31, 10, 29]
  Carga total: 135 s

Servidor 3:
  Tarefas (índices): [0, 7, 8, 13]
  Tempos: [12, 45, 60, 18]
  Carga total: 135 s

Makespan encontrado: 136 s
Limite inferior teórico: 136 s
O makespan encontrado atingiu o menor valor teoricamente possível.
